### **Data Integration**

The final merged dataset will be at the country-month level. Each row represents one country in one month: how many air passengers arrived in Singapore from that country that month, what the fuel price was, what that country's GDP per capita was, and other features. Example: "From Australia, in March 2022, 45,000 passengers arrived. Fuel was $3.80/gallon. GDP per capita was $54,000."

Loading cleaned data & beginning merging

In [3]:
import pandas as pd
import numpy as np

arrivals = pd.read_csv('../data/processed/arrivals_long.csv')
fuel = pd.read_csv('../data/processed/fuel_monthly.csv')
gdp = pd.read_csv('../data/processed/gdp_clean.csv')

print("Arrivals shape:", arrivals.shape)
print("Fuel shape:", fuel.shape)
print("GDP shape:", gdp.shape)

# Preview each
print("\nArrivals sample:")
print(arrivals.head(3))
print("\nFuel sample:")
print(fuel.head(3))

Arrivals shape: (11400, 3)
Fuel shape: (257, 2)
GDP shape: (5157, 4)

Arrivals sample:
                         DataSeries year_month  arrivals
0  Number Of Air Passenger Arrivals    2025-07   3042743
1                   South East Asia    2025-07   1173824
2                           Vietnam    2025-07    114128

Fuel sample:
  year_month  jet_fuel_usd_per_gallon
0    2005-01                  1.33550
1    2005-02                  1.33375
2    2005-03                  1.55225


Merge arrivals with fuel prices

In [4]:
import pandas as pd

#LOAD DATASETS (correct files)
arrivals_long = pd.read_csv("../data/processed/arrivals_long.csv")
fuel = pd.read_csv("../data/processed/fuel_monthly.csv")

print("Arrivals shape:", arrivals_long.shape)
print("Fuel shape:", fuel.shape)


#CONVERT DATE COLUMNS
arrivals_long['year_month'] = pd.to_datetime(arrivals_long['year_month'], errors='coerce')
fuel['year_month'] = pd.to_datetime(fuel['year_month'], errors='coerce')


arrivals_long = arrivals_long.dropna(subset=['year_month'])
fuel = fuel.dropna(subset=['year_month'])


#MERGE
df = arrivals_long.merge(fuel, on='year_month', how='left')


print(f"Rows before merge: {len(arrivals_long)}")
print(f"Rows after merge: {len(df)}")

if 'jet_fuel_usd_per_gallon' in df.columns:
    print("Missing fuel values:", df['jet_fuel_usd_per_gallon'].isnull().sum())
else:
    print("Fuel column not found — check fuel_monthly.csv structure")


print("Arrivals sample:", arrivals_long['year_month'].head(3).tolist())
print("Fuel sample:", fuel['year_month'].head(3).tolist())

print(df.head())

Arrivals shape: (11400, 3)
Fuel shape: (257, 2)
Rows before merge: 11400
Rows after merge: 11400
Missing fuel values: 6563
Arrivals sample: [Timestamp('2025-07-01 00:00:00'), Timestamp('2025-07-01 00:00:00'), Timestamp('2025-07-01 00:00:00')]
Fuel sample: [Timestamp('2005-01-01 00:00:00'), Timestamp('2005-02-01 00:00:00'), Timestamp('2005-03-01 00:00:00')]
                         DataSeries year_month  arrivals  \
0  Number Of Air Passenger Arrivals 2025-07-01   3042743   
1                   South East Asia 2025-07-01   1173824   
2                           Vietnam 2025-07-01    114128   
3                         Indonesia 2025-07-01    360848   
4                     Other Regions 2025-07-01     14828   

   jet_fuel_usd_per_gallon  
0                   2.2425  
1                   2.2425  
2                   2.2425  
3                   2.2425  
4                   2.2425  


**Add a year column for GDP merging**

In [5]:
df['year'] = df['year_month'].dt.year

**Building a country name mapping**

In [8]:
# The country names that exist in arrivals
print(sorted(arrivals_long['DataSeries'].unique().tolist()))

# The country names that exist in GDP
print(sorted(gdp['country'].unique().tolist()[:30]))

['Europe', 'France', 'Germany', 'Hong Kong', 'Indonesia', 'Japan', 'Mainland China', 'Malaysia', 'Middle East', 'North America', 'North East Asia', 'Number Of Air Passenger Arrivals', 'Oceania', 'Other Regions', 'Philippines', 'South Asia', 'South East Asia', 'Thailand', 'United Kingdom', 'Vietnam']
['Africa Eastern and Southern', 'Africa Western and Central', 'Arab World', 'Caribbean small states', 'Central Europe and the Baltics', 'Early-demographic dividend', 'East Asia & Pacific', 'East Asia & Pacific (IDA & IBRD countries)', 'East Asia & Pacific (excluding high income)', 'Euro area', 'Europe & Central Asia', 'Europe & Central Asia (IDA & IBRD countries)', 'Europe & Central Asia (excluding high income)', 'European Union', 'Fragile and conflict affected situations', 'Heavily indebted poor countries (HIPC)', 'High income', 'IBRD only', 'IDA & IBRD total', 'IDA blend', 'IDA only', 'IDA total', 'Late-demographic dividend', 'Latin America & Caribbean', 'Latin America & Caribbean (exclud

In [14]:
arrivals_long['year_month'] = pd.to_datetime(arrivals_long['year_month'], errors='coerce')
gdp['year'] = pd.to_numeric(gdp['year'], errors='coerce')

arrivals_long['year'] = arrivals_long['year_month'].dt.year

In [17]:
country_name_map = {
    'United States Of America': 'United States',
    'Korea, Republic Of': 'Korea, Rep.',
    'Viet Nam': 'Vietnam',
    'Hong Kong': 'Hong Kong SAR, China',
    'Mainland China': 'China',
    'Russia': 'Russian Federation',
    'Iran': 'Iran, Islamic Rep.',
    'Egypt': 'Egypt, Arab Rep.',
}

arrivals_long['country_gdp_key'] = arrivals_long['DataSeries'].replace(country_name_map)
arrivals_long['country'] = arrivals_long['country_gdp_key']

In [18]:
df = arrivals_long.merge(
    gdp[['country', 'year', 'gdp_per_capita']],
    on=['country', 'year'],
    how='left'
)

print(df.shape)
print("Missing GDP:", df['gdp_per_capita'].isna().sum())

(11400, 7)
Missing GDP: 8522


In [19]:
country_list = set(gdp['country'].unique())

df['is_country'] = df['country'].isin(country_list)

print(df['is_country'].value_counts())

is_country
True     7725
False    3675
Name: count, dtype: int64


In [20]:
df_countries = df[df['is_country'] == True]
df_regions = df[df['is_country'] == False]

print(df_countries.shape)
print(df_regions.shape)

(7725, 8)
(3675, 8)


In [21]:
df_final = df.copy()

print(df_final.shape)
print(df_final.head())

(11400, 8)
                         DataSeries year_month  arrivals  year  \
0  Number Of Air Passenger Arrivals 2025-07-01   3042743  2025   
1                   South East Asia 2025-07-01   1173824  2025   
2                           Vietnam 2025-07-01    114128  2025   
3                         Indonesia 2025-07-01    360848  2025   
4                     Other Regions 2025-07-01     14828  2025   

                    country_gdp_key                           country  \
0  Number Of Air Passenger Arrivals  Number Of Air Passenger Arrivals   
1                   South East Asia                   South East Asia   
2                           Vietnam                           Vietnam   
3                         Indonesia                         Indonesia   
4                     Other Regions                     Other Regions   

   gdp_per_capita  is_country  
0             NaN       False  
1             NaN       False  
2             NaN       False  
3             NaN        

In [23]:
df_countries = df[df['is_country']]
df_regions = df[~df['is_country']]
df_model = df_countries[df_countries['gdp_per_capita'].notna()]

In [24]:
df = df.merge(
    gdp[['country', 'year', 'gdp_per_capita']],
    left_on=['country_gdp_key', 'year'],
    right_on=['country', 'year'],
    how='left',
    suffixes=('', '_gdp')
)

# Drop the duplicate country column from GDP
df = df.drop(columns=['country_gdp'], errors='ignore')

print(f"Missing GDP values: {df['gdp_per_capita'].isnull().sum()}")
print(f"GDP coverage: {df['gdp_per_capita'].notna().mean():.1%}")

Missing GDP values: 8522
GDP coverage: 25.2%


In [25]:
df_model = df[
    (df['gdp_per_capita'].notna())
].copy()

df_countries = df[df['is_country'] == True]
df_regions = df[df['is_country'] == False]

**Add route-level features from OurAirports**

For each country in the arrivals dataset: the average distance from Singapore to that country's main airport(s), and the LCC market share on the corridor.

First, calculating distances using the Haversine formula:

In [32]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

def haversine_km(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two points on Earth."""
    R = 6371
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))


SIN_LAT, SIN_LON = 1.3644, 103.9915

airports = pd.read_csv('../data/processed/airports.csv')
changi_routes = pd.read_csv('../data/processed/routes.csv')
airline_types = pd.read_csv('../data/processed/airline_classifications_clean.csv')


routes_with_airports = changi_routes.merge(
    airports[['iata_code', 'iso_country', 'latitude_deg', 'longitude_deg']],
    left_on='destination_airport_iata',
    right_on='iata_code',
    how='left'
)

routes_with_airports = routes_with_airports.drop(columns=['iata_code'], errors='ignore')


routes_with_airports['distance_km'] = routes_with_airports.apply(
    lambda row: haversine_km(
        SIN_LAT, SIN_LON,
        row['latitude_deg'], row['longitude_deg']
    ) if pd.notna(row['latitude_deg']) else np.nan,
    axis=1
)


routes_with_airports = routes_with_airports.merge(
    airline_types[['airline_iata', 'carrier_type']],
    on='airline_iata',
    how='left'
)

routes_with_airports['carrier_type'] = routes_with_airports['carrier_type'].fillna('Unknown')


print("Shape:", routes_with_airports.shape)
print("Missing distances:", routes_with_airports['distance_km'].isna().sum())
print("Carrier types:", routes_with_airports['carrier_type'].value_counts().head())
print(routes_with_airports.head())

Shape: (374, 9)
Missing distances: 0
Carrier types: carrier_type
FSC        174
LCC        170
Unknown     30
Name: count, dtype: int64
  airline_iata destination_airport_iata destination_city destination_country  \
0           SQ                      SYD           Sydney           Australia   
1           SQ                      MEL        Melbourne           Australia   
2           SQ                      BNE         Brisbane           Australia   
3           SQ                      PER            Perth           Australia   
4           SQ                      ADL         Adelaide           Australia   

  iso_country  latitude_deg  longitude_deg  distance_km carrier_type  
0          AU    -33.946098     151.177002  6294.727208          FSC  
1          AU    -37.670732     144.837898  6033.904250          FSC  
2          AU    -27.384199     153.117004  6143.832010          FSC  
3          AU    -31.940300     115.967003  3912.358154          FSC  
4          AU    -34.947512 

Aggregating to country level:

In [34]:
# For each destination country: average distance and LCC share
country_route_features = routes_with_airports.groupby('iso_country').agg(
    avg_distance_km=('distance_km', 'mean'),
    total_routes=('airline_iata', 'count'),
    lcc_routes=('carrier_type', lambda x: (x == 'LCC').sum())
).reset_index()

country_route_features['lcc_share'] = (
    country_route_features['lcc_routes'] / country_route_features['total_routes']
)

print(country_route_features.head(10))
print(f"Countries with route data: {len(country_route_features)}")

  iso_country  avg_distance_km  total_routes  lcc_routes  lcc_share
0          AE      5860.990832             5           1   0.200000
1          AT      9703.198494             1           0   0.000000
2          AU      5266.401892            19           6   0.315789
3          BD      2896.717931             3           1   0.333333
4          BE     10553.931665             1           0   0.000000
5          BH      6330.626953             1           0   0.000000
6          BN      1277.627331             2           0   0.000000
7          BT      3285.279151             2           1   0.500000
8          CA     12811.713806             2           0   0.000000
9          CH     10304.399725             2           0   0.000000
Countries with route data: 48


**Mapping from country ISO code to country names**

In [41]:
iso_to_country = {
    'AE': 'United Arab Emirates',
    'AT': 'Austria',
    'AU': 'Australia',
    'BD': 'Bangladesh',
    'BE': 'Belgium',
    'BH': 'Bahrain',
    'BN': 'Brunei',
    'BT': 'Bhutan',
    'CA': 'Canada',
    'CH': 'Switzerland',
    'CN': 'China',
    'DE': 'Germany',
    'DK': 'Denmark',
    'EG': 'Egypt',
    'ES': 'Spain',
    'FR': 'France',
    'GB': 'United Kingdom',
    'HK': 'Hong Kong SAR, China',
    'ID': 'Indonesia',
    'IN': 'India',
    'IT': 'Italy',
    'JP': 'Japan',
    'KR': 'Korea, Rep.',
    'LA': 'Lao PDR',
    'LK': 'Sri Lanka',
    'MY': 'Malaysia',
    'NL': 'Netherlands',
    'NZ': 'New Zealand',
    'PH': 'Philippines',
    'QA': 'Qatar',
    'SA': 'Saudi Arabia',
    'SG': 'Singapore',
    'TH': 'Thailand',
    'US': 'United States',
    'VN': 'Vietnam',
    'ET': 'Ethiopia',
    'FI': 'Finland',
    'FJ': 'Fiji',
    'GR': 'Greece',
    'KH': 'Cambodia',
    'MM': 'Myanmar',
    'MN': 'Mongolia',
    'MO': 'Macao SAR, China',
    'MV': 'Maldives',
    'NC': 'New Caledonia',
    'NP': 'Nepal',
    'OM': 'Oman',
    'PG': 'Papua New Guinea',
    'TR': 'Turkiye',
    'TW': 'Taiwan, China',
    'ZA': 'South Africa'
}

In [42]:
country_route_features['country'] = country_route_features['iso_country'].map(iso_to_country)

**Merging with arrivals / GDP**

In [44]:
df_final = country_route_features.merge(
    df[['country', 'gdp_per_capita']].drop_duplicates(),
    on='country',
    how='left'
)

**Defining our target variable**

Our model will predict: for a given country corridor in a given month, what is the year-over-year percentage change in passenger arrivals?
This cell constructs a unified panel dataset by merging international air arrivals with external macroeconomic and transport cost indicators, followed by transformation into a regression-ready format.

In [58]:
import pandas as pd
import numpy as np

arrivals = pd.read_csv('../data/processed/arrivals_long.csv')
fuel = pd.read_csv('../data/processed/fuel_monthly.csv')
gdp = pd.read_csv('../data/processed/gdp_clean.csv')

arrivals = arrivals.rename(columns={
    'DataSeries': 'country'
})

arrivals['year_month'] = pd.to_datetime(arrivals['year_month'])
fuel['year_month'] = pd.to_datetime(fuel['year_month'])

arrivals['year'] = arrivals['year_month'].dt.year

df = arrivals.merge(fuel, on='year_month', how='left')

df = df.merge(
    gdp[['country', 'year', 'gdp_per_capita']],
    on=['country', 'year'],
    how='left'
)

df = df.sort_values(['country', 'year_month']).reset_index(drop=True)

df['log_arrivals'] = np.log(df['arrivals'])
df['log_arrivals_diff'] = df.groupby('country')['log_arrivals'].diff()

df_model = df.dropna(subset=[
    'log_arrivals_diff',
    'jet_fuel_usd_per_gallon',
    'gdp_per_capita'
])

print("Final shape:", df.shape)
print("Model rows:", len(df_model))
print(df_model['log_arrivals_diff'].describe())

Final shape: (11400, 8)
Model rows: 2398
count    2398.000000
mean        0.002831
std         0.396759
min        -6.101167
25%        -0.072103
50%         0.012158
75%         0.102939
max         1.627227
Name: log_arrivals_diff, dtype: float64


Categorization risk label for visualization purposes

In [62]:
df_model = df_model.copy()

def risk_label(log_growth):
    if log_growth < -0.10:
        return 'High decline'
    elif log_growth < -0.02:
        return 'Moderate decline'
    elif log_growth < 0.02:
        return 'Stable'
    else:
        return 'Growth'

df_model['arrival_trend'] = df_model['log_arrivals_diff'].apply(risk_label)

print(df_model['arrival_trend'].value_counts())

arrival_trend
Growth              1131
Moderate decline     502
High decline         459
Stable               306
Name: count, dtype: int64
